# 18.4 CI/CD与自动化部署 (CI/CD & Automated Deployment)

> 🕐 预估学习时间：60分钟

本节介绍机器学习场景下的CI/CD与自动化部署实践。与传统软件CI/CD不同，ML流水线需要处理数据依赖、模型版本管理、训练-评估-部署的完整闭环。

**学习主题：**

1. ML CI/CD核心概念与传统CI/CD的区别
2. 模型评估流水线（ModelPipeline类）
3. A/B测试与金丝雀发布（ABTestFramework类）
4. 自动化重训练触发器（RetrainingTrigger类）
5. 模型治理与合规（ModelGovernance类）

## 1. ML CI/CD核心概念

传统软件CI/CD关注代码变更的构建、测试和部署，而ML CI/CD需要额外考虑数据依赖和模型版本管理。

**与传统CI/CD的关键区别：**

- **数据依赖**：模型性能不仅取决于代码，还取决于训练数据的质量和分布
- **模型版本**：需要同时管理代码版本、数据版本和模型版本
- **训练-评估-部署流水线**：完整的ML流水线包含训练、评估、验证、注册、部署多个阶段
- **持续训练（CT）**：ML特有的概念，模型需要随数据变化定期重训练

**ML CI/CD流水线典型阶段：**

1. 数据验证（Data Validation）
2. 模型训练（Model Training）
3. 模型评估（Model Evaluation）
4. 模型验证（Model Validation）
5. 模型注册（Model Registry）
6. 模型部署（Model Deployment）
7. 监控反馈（Monitoring & Feedback）

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== ML CI/CD 核心概念演示 ===')

# 传统CI/CD流水线阶段
traditional_cicd = [
    '代码提交 (Code Commit)',
    '构建 (Build)',
    '单元测试 (Unit Test)',
    '集成测试 (Integration Test)',
    '部署 (Deploy)',
]

# ML CI/CD流水线阶段
ml_cicd = [
    '数据验证 (Data Validation)',
    '模型训练 (Model Training)',
    '模型评估 (Model Evaluation)',
    '模型验证 (Model Validation)',
    '模型注册 (Model Registry)',
    '模型部署 (Model Deployment)',
    '监控反馈 (Monitoring)',
]

print('\n传统CI/CD流水线阶段:')
for i, stage in enumerate(traditional_cicd, 1):
    print(f'  {i}. {stage}')

print('\nML CI/CD流水线阶段:')
for i, stage in enumerate(ml_cicd, 1):
    print(f'  {i}. {stage}')

# ML CI/CD的三个版本管理维度
print('\nML版本管理三维度:')
dimensions = OrderedDict([
    ('代码版本', '管理训练代码、配置、流水线定义'),
    ('数据版本', '管理训练数据集的版本和血缘'),
    ('模型版本', '管理模型构件、指标、元数据'),
])
for dim, desc in dimensions.items():
    print(f'  {dim}: {desc}')

# 模拟数据漂移对模型性能的影响
print('\n=== 数据漂移影响模拟 ===')
n_samples = 1000
train_mean, train_std = 0.0, 1.0
train_data = torch.randn(n_samples) * train_std + train_mean

drift_mean, drift_std = 0.5, 1.2
prod_data = torch.randn(n_samples) * drift_std + drift_mean

threshold = train_mean + 2 * train_std
train_anomaly_rate = (train_data.abs() > threshold).float().mean().item()
prod_anomaly_rate = (prod_data.abs() > threshold).float().mean().item()

print(f'训练分布: mean={train_mean}, std={train_std}')
print(f'生产分布: mean={drift_mean}, std={drift_std}')
print(f'异常阈值: {threshold:.2f}')
print(f'训练数据异常率: {train_anomaly_rate:.2%}')
print(f'生产数据异常率: {prod_anomaly_rate:.2%}')
print(f'漂移导致的异常率上升: {prod_anomaly_rate - train_anomaly_rate:.2%}')

print(f'\nKey: ML CI/CD在传统CI/CD基础上增加了数据验证、模型注册和监控反馈阶段，需要管理代码、数据、模型三个维度的版本')

## 2. 模型评估流水线

ModelPipeline类模拟完整的ML流水线，包含训练→评估→验证→注册→部署的完整流程。

**关键组件：**

- **训练阶段**：使用训练数据拟合模型参数
- **评估阶段**：在验证集上计算性能指标
- **验证阶段**：检查模型是否满足业务阈值
- **注册阶段**：将通过验证的模型注册到模型仓库
- **部署阶段**：将模型部署到生产环境

每个阶段都有明确的通过条件和日志记录，确保流水线的可追溯性。

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 模型评估流水线 ModelPipeline ===')

class ModelPipeline:
    '''完整的ML模型流水线：训练→评估→验证→注册→部署'''

    def __init__(self, model_name, input_dim=10, output_dim=2):
        self.model_name = model_name
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.model = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim),
        )
        self.version = 'v1.0.0'
        self.stage_results = OrderedDict()
        self.model_registry = []
        self.deployed = False

    def generate_data(self, n_samples=500):
        '''生成模拟训练数据'''
        X = torch.randn(n_samples, self.input_dim)
        logits = X[:, 0] * 2 + X[:, 1] - X[:, 2] * 0.5 + X[:, 3] * 1.5
        y = (logits > 0).long()
        return X, y

    def train(self, epochs=50, lr=0.01):
        '''训练阶段'''
        print(f'\n[训练阶段] 训练模型 {self.model_name} ({self.version})')
        X, y = self.generate_data(500)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        losses = []
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = self.model(X)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        final_loss = losses[-1]
        self.stage_results['train'] = OrderedDict([
            ('final_loss', final_loss),
            ('epochs', epochs),
            ('samples', len(y)),
        ])
        print(f'  训练完成: {epochs}轮, 最终损失={final_loss:.4f}')
        return final_loss

    def evaluate(self):
        '''评估阶段：在验证集上计算指标'''
        print('\n[评估阶段] 在验证集上评估模型')
        X, y = self.generate_data(200)
        self.model.eval()
        with torch.no_grad():
            output = self.model(X)
            pred = output.argmax(dim=1)
            accuracy = (pred == y).float().mean().item()

        self.stage_results['evaluate'] = OrderedDict([
            ('accuracy', accuracy),
            ('val_samples', len(y)),
        ])
        print(f'  验证准确率: {accuracy:.2%} (样本数={len(y)})')
        return accuracy

    def validate(self, min_accuracy=0.85):
        '''验证阶段：检查是否满足业务阈值'''
        print(f'\n[验证阶段] 检查模型是否满足业务阈值 ({min_accuracy:.0%})')
        acc = self.stage_results['evaluate']['accuracy']
        passed = acc >= min_accuracy
        checks = OrderedDict([
            ('accuracy_threshold', acc >= min_accuracy),
            ('no_nan', not math.isnan(acc)),
            ('model_size_ok', sum(p.numel() for p in self.model.parameters()) < 10000),
        ])
        self.stage_results['validate'] = OrderedDict([
            ('passed', passed),
            ('checks', checks),
        ])
        for check_name, result in checks.items():
            status = '通过' if result else '失败'
            print(f'  {check_name}: {status}')
        result_text = '通过' if passed else '失败'
        print(f'  验证结果: {result_text}')
        return passed

    def register(self):
        '''注册阶段：将模型注册到模型仓库'''
        print('\n[注册阶段] 注册模型到模型仓库')
        validate_result = self.stage_results.get('validate', {})
        if not validate_result.get('passed', False):
            print('  注册失败: 模型未通过验证')
            return False

        params = sum(p.numel() for p in self.model.parameters())
        model_info = OrderedDict([
            ('name', self.model_name),
            ('version', self.version),
            ('accuracy', self.stage_results['evaluate']['accuracy']),
            ('params', params),
            ('stage', 'registered'),
        ])
        self.model_registry.append(model_info)
        self.stage_results['register'] = model_info
        print(f'  注册成功: {self.model_name} {self.version}')
        print(f'  模型参数量: {params}')
        return True

    def deploy(self):
        '''部署阶段：将模型部署到生产环境'''
        print('\n[部署阶段] 部署模型到生产环境')
        if 'register' not in self.stage_results:
            print('  部署失败: 模型未注册')
            return False

        self.deployed = True
        endpoint = f'/api/v1/models/{self.model_name}/predict'
        self.stage_results['deploy'] = OrderedDict([
            ('endpoint', endpoint),
            ('status', 'serving'),
            ('replicas', 3),
        ])
        print(f'  部署成功: {endpoint}')
        print('  副本数: 3')
        return True

    def run_full_pipeline(self):
        '''运行完整流水线'''
        print('=' * 50)
        print(f'运行完整ML流水线: {self.model_name}')
        print('=' * 50)
        self.train()
        self.evaluate()
        self.validate()
        self.register()
        self.deploy()
        print('\n' + '=' * 50)
        print('流水线执行完成，阶段摘要:')
        for stage, result in self.stage_results.items():
            print(f'  {stage}: {result}')
        return self.deployed

# 运行完整流水线
pipeline = ModelPipeline('text_classifier', input_dim=10, output_dim=2)
success = pipeline.run_full_pipeline()

print(f'\nKey: ModelPipeline封装了训练→评估→验证→注册→部署的完整流程，每个阶段都有明确的通过条件和日志记录')

## 3. A/B测试与金丝雀发布

A/B测试用于比较不同模型版本在生产环境中的实际表现，金丝雀发布是一种渐进式部署策略。

**A/B测试关键要素：**

- **流量分配**：将用户请求按比例分配到不同模型版本
- **指标收集**：收集每个模型版本的关键业务指标
- **统计显著性检验**：判断模型差异是否显著（使用z检验或t检验）
- **金丝雀发布**：从小比例流量开始，逐步扩大部署范围

**统计显著性检验**通过计算p值来判断两个模型版本的指标差异是否由随机波动引起。通常p值小于0.05认为差异显著。

In [ ]:
import torch
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== A/B测试与金丝雀发布 ABTestFramework ===')

class ABTestFramework:
    '''A/B测试框架：流量分配、指标收集、统计检验、金丝雀发布'''

    def __init__(self):
        self.experiments = {}
        self.canary_stages = [0.05, 0.10, 0.25, 0.50, 1.00]

    def create_experiment(self, name, model_a, model_b, traffic_split=0.5):
        '''创建A/B测试实验'''
        self.experiments[name] = OrderedDict([
            ('model_a', model_a),
            ('model_b', model_b),
            ('traffic_split', traffic_split),
            ('metrics_a', []),
            ('metrics_b', []),
            ('status', 'running'),
        ])
        print(f'创建实验 [{name}]: 模型A={model_a}, 模型B={model_b}, 流量分配={traffic_split:.0%}')

    def simulate_traffic(self, exp_name, n_requests=1000):
        '''模拟流量分配和指标收集'''
        exp = self.experiments[exp_name]
        split = exp['traffic_split']

        conv_rate_a = 0.10
        conv_rate_b = 0.12

        assignments = torch.rand(n_requests) < split
        n_b = assignments.sum().item()
        n_a = n_requests - n_b

        outcomes_a = (torch.rand(n_a) < conv_rate_a).float()
        outcomes_b = (torch.rand(n_b) < conv_rate_b).float()

        exp['metrics_a'].extend(outcomes_a.tolist())
        exp['metrics_b'].extend(outcomes_b.tolist())

        print(f'\n实验 [{exp_name}] 流量模拟 ({n_requests} 请求):')
        print(f'  模型A流量: {n_a} ({n_a/n_requests:.1%})')
        print(f'  模型B流量: {n_b} ({n_b/n_requests:.1%})')

        return n_a, n_b

    def compute_statistics(self, exp_name):
        '''计算统计指标和显著性检验'''
        exp = self.experiments[exp_name]
        metrics_a = torch.tensor(exp['metrics_a'])
        metrics_b = torch.tensor(exp['metrics_b'])

        n_a = len(metrics_a)
        n_b = len(metrics_b)
        if n_a == 0 or n_b == 0:
            print('样本量不足，无法计算统计量')
            return None

        p_a = metrics_a.mean().item()
        p_b = metrics_b.mean().item()

        p_pool = (metrics_a.sum() + metrics_b.sum()).item() / (n_a + n_b)
        se = math.sqrt(p_pool * (1 - p_pool) * (1.0/n_a + 1.0/n_b))

        if se == 0:
            z_score = 0.0
            p_value = 1.0
        else:
            z_score = (p_b - p_a) / se
            p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(z_score) / math.sqrt(2))))

        stats = OrderedDict([
            ('n_a', n_a),
            ('n_b', n_b),
            ('conv_rate_a', p_a),
            ('conv_rate_b', p_b),
            ('uplift', p_b - p_a),
            ('z_score', z_score),
            ('p_value', p_value),
            ('significant', p_value < 0.05),
        ])

        print(f'\n实验 [{exp_name}] 统计分析:')
        print(f'  模型A转化率: {p_a:.4f} (n={n_a})')
        print(f'  模型B转化率: {p_b:.4f} (n={n_b})')
        print(f'  提升幅度: {p_b - p_a:.4f} ({(p_b - p_a)/p_a:.2%})')
        print(f'  Z分数: {z_score:.4f}')
        print(f'  P值: {p_value:.6f}')
        sig_text = '显著' if p_value < 0.05 else '不显著'
        print(f'  统计显著性: {sig_text} (阈值=0.05)')

        return stats

    def canary_release(self, model_name, target_traffic=1.0):
        '''金丝雀渐进发布'''
        print(f'\n=== 金丝雀发布: {model_name} ===')
        current_traffic = 0.0
        stage_num = 0

        for stage_traffic in self.canary_stages:
            if stage_traffic > target_traffic:
                break
            stage_num += 1
            error_rate = torch.rand(1).item() * 0.02
            latency = 50 + torch.rand(1).item() * 20
            healthy = error_rate < 0.05 and latency < 100

            print(f'  阶段{stage_num}: 流量={stage_traffic:.0%}, 错误率={error_rate:.2%}, 延迟={latency:.0f}ms')

            if not healthy:
                print('  [失败] 健康检查失败，自动回滚!')
                return False

            current_traffic = stage_traffic
            print('  [通过] 健康检查通过，继续下一阶段')

        print(f'  [完成] 金丝雀发布完成: {model_name} 已承接 {current_traffic:.0%} 流量')
        return True

# 运行A/B测试
framework = ABTestFramework()
framework.create_experiment('exp_001', 'model_v1', 'model_v2', traffic_split=0.5)
framework.simulate_traffic('exp_001', n_requests=5000)
stats = framework.compute_statistics('exp_001')

# 运行金丝雀发布
framework.canary_release('model_v2')

print(f'\nKey: A/B测试通过z检验判断模型差异的统计显著性，金丝雀发布通过渐进式流量扩大降低部署风险')

## 4. 自动化重训练触发器

RetrainingTrigger类实现基于多种信号的自动重训练触发机制。

**触发条件类型：**

- **数据漂移**：特征分布或标签分布发生变化（使用KL散度或PSI检测）
- **性能下降**：模型在线性能低于预设阈值
- **时间间隔**：定期重训练（如每周、每月）
- **数据量积累**：新增数据达到一定量

**数据漂移检测**常用方法包括Population Stability Index (PSI)和KL散度。PSI > 0.2通常表示显著漂移。

In [ ]:
import torch
import math
import time
from collections import OrderedDict

torch.manual_seed(42)

print('=== 自动化重训练触发器 RetrainingTrigger ===')

class RetrainingTrigger:
    '''自动化重训练触发器：基于数据漂移、性能下降、时间间隔'''

    def __init__(self):
        self.triggers = []
        self.baseline_distribution = None
        self.last_train_time = time.time()
        self.last_performance = 0.92
        self.new_data_count = 0

    def set_baseline(self, data):
        '''设置基线数据分布'''
        self.baseline_distribution = self._compute_histogram(data)
        print(f'基线分布已设置 (样本数={len(data)})')

    def _compute_histogram(self, data, n_bins=10):
        '''计算数据直方图分布'''
        data_tensor = data if isinstance(data, torch.Tensor) else torch.tensor(data)
        hist = torch.histc(data_tensor.float(), bins=n_bins)
        hist = hist / hist.sum()
        hist = torch.clamp(hist, min=1e-8)
        return hist

    def compute_psi(self, new_data):
        '''计算Population Stability Index (PSI)'''
        if self.baseline_distribution is None:
            return 0.0
        new_hist = self._compute_histogram(new_data)
        psi = ((new_hist - self.baseline_distribution) *
               torch.log(new_hist / self.baseline_distribution)).sum().item()
        return psi

    def check_data_drift(self, new_data, threshold=0.2):
        '''检查数据漂移'''
        psi = self.compute_psi(new_data)
        drifted = psi > threshold
        result = OrderedDict([
            ('trigger', 'data_drift'),
            ('psi', psi),
            ('threshold', threshold),
            ('triggered', drifted),
        ])
        self.triggers.append(result)
        status = '触发' if drifted else '未触发'
        print(f'  数据漂移检查: PSI={psi:.4f} (阈值={threshold}), {status}')
        return drifted

    def check_performance_drop(self, current_performance, threshold=0.85):
        '''检查性能下降'''
        dropped = current_performance < threshold
        result = OrderedDict([
            ('trigger', 'performance_drop'),
            ('current_perf', current_performance),
            ('threshold', threshold),
            ('triggered', dropped),
        ])
        self.triggers.append(result)
        status = '触发' if dropped else '未触发'
        print(f'  性能下降检查: 当前={current_performance:.4f} (阈值={threshold}), {status}')
        return dropped

    def check_time_interval(self, interval_seconds=3600):
        '''检查时间间隔'''
        elapsed = time.time() - self.last_train_time
        elapsed_simulated = elapsed + 4000
        triggered = elapsed_simulated > interval_seconds
        result = OrderedDict([
            ('trigger', 'time_interval'),
            ('elapsed', elapsed_simulated),
            ('interval', interval_seconds),
            ('triggered', triggered),
        ])
        self.triggers.append(result)
        status = '触发' if triggered else '未触发'
        print(f'  时间间隔检查: 已过={elapsed_simulated:.0f}秒 (间隔={interval_seconds}秒), {status}')
        return triggered

    def check_data_volume(self, new_count, threshold=10000):
        '''检查数据量积累'''
        self.new_data_count += new_count
        triggered = self.new_data_count >= threshold
        result = OrderedDict([
            ('trigger', 'data_volume'),
            ('new_count', self.new_data_count),
            ('threshold', threshold),
            ('triggered', triggered),
        ])
        self.triggers.append(result)
        status = '触发' if triggered else '未触发'
        print(f'  数据量检查: 新增={self.new_data_count} (阈值={threshold}), {status}')
        return triggered

    def evaluate_all(self, new_data, current_performance, new_data_count):
        '''评估所有触发条件'''
        print('\n--- 重训练触发器评估 ---')
        drift = self.check_data_drift(new_data)
        perf = self.check_performance_drop(current_performance)
        time_trig = self.check_time_interval(interval_seconds=3600)
        vol = self.check_data_volume(new_data_count, threshold=10000)

        any_triggered = drift or perf or time_trig or vol
        retrain_text = '需要重训练' if any_triggered else '无需重训练'
        print(f'\n  综合结果: {retrain_text}')
        return any_triggered

    def get_trigger_history(self):
        '''获取触发历史'''
        return self.triggers

# 演示重训练触发器
trigger = RetrainingTrigger()

# 设置基线分布
baseline_data = torch.randn(5000)
trigger.set_baseline(baseline_data)

# 场景1：轻微漂移
print('\n--- 场景1: 轻微数据漂移 ---')
slight_drift = torch.randn(5000) * 1.05 + 0.05
trigger.evaluate_all(slight_drift, current_performance=0.90, new_data_count=3000)

# 场景2：严重漂移 + 性能下降
print('\n--- 场景2: 严重数据漂移 + 性能下降 ---')
severe_drift = torch.randn(5000) * 1.5 + 0.8
trigger.evaluate_all(severe_drift, current_performance=0.80, new_data_count=8000)

# 查看触发历史
print('\n=== 触发历史摘要 ===')
for i, t in enumerate(trigger.get_trigger_history(), 1):
    triggered = t.get('triggered', False)
    status = '[触发]' if triggered else '[未触发]'
    trigger_name = t['trigger']
    print(f'  {i}. {trigger_name}: {status}')

print(f'\nKey: RetrainingTrigger通过PSI检测数据漂移、性能阈值、时间间隔和数据量积累四个维度自动触发重训练')

## 5. 模型治理与合规

ModelGovernance类提供模型审计、权限管理和合规检查功能。

**治理要素：**

- **审计日志**：记录模型全生命周期的操作（谁、何时、做了什么）
- **权限管理**：控制谁可以训练、评估、部署模型（RBAC角色权限模型）
- **合规检查**：确保模型满足业务、安全和法规要求
- **模型卡片**：记录模型的用途、限制、性能、训练数据等信息

模型治理是MLOps成熟度的重要标志，确保模型的可追溯性、可审计性和合规性。

In [ ]:
import torch
import math
import time
from collections import OrderedDict

torch.manual_seed(42)

print('=== 模型治理与合规 ModelGovernance ===')

class ModelGovernance:
    '''模型治理：审计日志、权限管理、合规检查'''

    def __init__(self):
        self.audit_logs = []
        self.model_cards = {}
        self.roles = OrderedDict([
            ('data_scientist', ['train', 'evaluate', 'view']),
            ('ml_engineer', ['train', 'evaluate', 'register', 'deploy', 'view']),
            ('admin', ['train', 'evaluate', 'register', 'deploy', 'view', 'delete', 'manage_users']),
            ('auditor', ['view', 'view_logs']),
        ])
        self.users = OrderedDict([
            ('alice', 'data_scientist'),
            ('bob', 'ml_engineer'),
            ('carol', 'admin'),
            ('dave', 'auditor'),
        ])

    def log_action(self, user, action, model_name, details=''):
        '''记录审计日志'''
        log_entry = OrderedDict([
            ('timestamp', time.time()),
            ('user', user),
            ('action', action),
            ('model', model_name),
            ('details', details),
        ])
        self.audit_logs.append(log_entry)

    def check_permission(self, user, action):
        '''检查用户权限'''
        role = self.users.get(user)
        if role is None:
            return False
        permissions = self.roles.get(role, [])
        return action in permissions

    def authorize_action(self, user, action, model_name):
        '''授权并记录操作'''
        allowed = self.check_permission(user, action)
        role = self.users.get(user, 'unknown')
        if allowed:
            self.log_action(user, action, model_name, 'authorized')
            print(f'  [OK] {user} ({role}) 执行 {action} on {model_name}: 已授权')
        else:
            self.log_action(user, action, model_name, 'DENIED')
            print(f'  [DENY] {user} ({role}) 执行 {action} on {model_name}: 权限拒绝')
        return allowed

    def create_model_card(self, model_name, info):
        '''创建模型卡片'''
        self.model_cards[model_name] = OrderedDict(info)
        self.log_action('system', 'create_card', model_name, 'model card created')
        print(f'  模型卡片已创建: {model_name}')

    def compliance_check(self, model_name):
        '''合规检查清单'''
        card = self.model_cards.get(model_name, {})
        checks = OrderedDict([
            ('has_model_card', model_name in self.model_cards),
            ('has_training_data_desc', 'training_data' in card),
            ('has_performance_metrics', 'performance' in card),
            ('has_limitations', 'limitations' in card),
            ('has_bias_assessment', 'bias_assessment' in card),
            ('has_intended_use', 'intended_use' in card),
        ])
        passed = all(checks.values())
        print(f'\n  合规检查 [{model_name}]:')
        for check_name, result in checks.items():
            status = '[OK]' if result else '[FAIL]'
            print(f'    {status} {check_name}')
        compliance_text = '通过' if passed else '不通过'
        print(f'  合规结果: {compliance_text}')
        return passed

    def get_audit_trail(self, model_name=None):
        '''获取审计追踪'''
        if model_name:
            return [log for log in self.audit_logs if log['model'] == model_name]
        return self.audit_logs

# 演示模型治理
gov = ModelGovernance()

print('--- 权限管理演示 ---')
gov.authorize_action('alice', 'deploy', 'text_classifier')
gov.authorize_action('bob', 'deploy', 'text_classifier')
gov.authorize_action('dave', 'train', 'text_classifier')
gov.authorize_action('carol', 'manage_users', 'text_classifier')

print('\n--- 模型卡片创建 ---')
gov.create_model_card('text_classifier', {
    'version': 'v1.0.0',
    'intended_use': '文本情感分类，适用于社交媒体评论',
    'training_data': '100万条中文评论，2024年1月收集',
    'performance': {'accuracy': 0.92, 'f1': 0.89},
    'limitations': '不适用于长文本和专业领域文本',
    'bias_assessment': '已在性别和年龄维度评估，无显著偏差',
})

print('\n--- 合规检查 ---')
gov.compliance_check('text_classifier')

print('\n=== 审计日志追踪 ===')
audit_trail = gov.get_audit_trail()
for i, log in enumerate(audit_trail, 1):
    action = log['action']
    user = log['user']
    model = log['model']
    details = log['details']
    print(f'  {i}. [{action}] {user} -> {model} ({details})')

print(f'\nKey: ModelGovernance通过RBAC权限模型、审计日志、模型卡片和合规检查清单实现模型全生命周期的治理与合规')

## 📝 课后思考题

1. ML CI/CD与传统软件CI/CD有哪些核心区别？这些区别如何影响流水线设计？
2. 在A/B测试中，如何确定统计显著性？样本量不足时应该如何处理？
3. 设计一个重训练触发系统时，如何平衡漂移检测的敏感性与误报率？
4. 模型治理中的审计日志应该记录哪些关键信息？如何保证日志的不可篡改性？